# Rule: **build_energy_totals**

**Outputs**

- resources/`transformation_output_coke.csv`
- resources/`energy_totals.csv`
- resources/`co2_totals.csv`
- resources/`transport_data.csv`
- resources/`district_heat_share.csv`
- resources/`heating_efficiencies.csv`

In [ ]:
######################################## Parameters

### Run
name = 'case_SectorCoupled'
prefix = ''

### Network
horizon = '2040'

In [ ]:
##### Import packages
import pypsa
import pandas as pd
import cartopy.crs as ccrs
import geopandas as gpd
import matplotlib.pyplot as plt
import os 
import sys
import numpy as np


##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp


##### Read params.yaml
params = xp.read_params('../params.yaml')


##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

## `transformation_output_coke.csv`

Load the file and preview its content.

In [ ]:
file = "transformation_output_coke.csv"

df_coke = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

df_coke.head()

## `energy_totals.csv`

Load the file and preview its content.

In [ ]:
file = "energy_totals.csv"

df_energy_totals = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

df_energy_totals.head()

Visualization for `country == "ES"`.

### `residential`

In [ ]:
#################### Parameters
country = "ES"
year_start = 1990
year_end = 2024



#################### 4-panel bar chart: space, water, cooking, and total residential

if "country" not in df_energy_totals.columns or "year" not in df_energy_totals.columns:
    raise KeyError("The DataFrame must contain 'country' and 'year' columns.")

df_es = df_energy_totals[df_energy_totals["country"] == country].copy()
if df_es.empty:
    raise ValueError(f"No rows found for country == '{country}'.")

df_es = df_es[(df_es["year"] >= year_start) & (df_es["year"] <= year_end)].copy()
if df_es.empty:
    raise ValueError(f"No rows found for {country} in the selected period [{year_start}, {year_end}].")

pairs = [
    ("total residential space", "electricity residential space", "Space"),
    ("total residential water", "electricity residential water", "Water"),
    ("total residential cooking", "electricity residential cooking", "Cooking"),
    ("total residential", "electricity residential", "Residential total"),
]

required_cols = [c for pair in pairs for c in pair[:2]]
missing_cols = [c for c in required_cols if c not in df_es.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

plot_df = (
    df_es.groupby("year", dropna=False)[required_cols]
    .sum()
    .sort_index()
)

x = np.arange(len(plot_df.index))
width = 0.38

# 70% overlap between Total and Electricity bars
overlap = 0.70
center_shift = width * (1 - overlap) / 2

# Shared y-scale across all panels
ymax = max(plot_df[required_cols].max()) * 1.05

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (total_col, elec_col, title) in zip(axes, pairs):
    ax.bar(x - center_shift, plot_df[total_col].values, width=width, color="#4E79A7", alpha=0.8, label="Total")
    ax.bar(x + center_shift, plot_df[elec_col].values, width=width, color="#F28E2B", alpha=0.8, label="Electricity")

    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("Value")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index.astype(str), rotation=45)
    ax.set_ylim(0, ymax)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

fig.suptitle(f"{country} residential demand components by year", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### `services`

In [ ]:
#################### Parameters
country = "ES"
year_start = 1990
year_end = 2024



#################### 4-panel bar chart: space, water, cooking, and total services

if "country" not in df_energy_totals.columns or "year" not in df_energy_totals.columns:
    raise KeyError("The DataFrame must contain 'country' and 'year' columns.")

df_country = df_energy_totals[df_energy_totals["country"] == country].copy()
if df_country.empty:
    raise ValueError(f"No rows found for country == '{country}'.")

df_country = df_country[(df_country["year"] >= year_start) & (df_country["year"] <= year_end)].copy()
if df_country.empty:
    raise ValueError(f"No rows found for {country} in the selected period [{year_start}, {year_end}].")

pairs = [
    ("total services space", "electricity services space", "Space"),
    ("total services water", "electricity services water", "Water"),
    ("total services cooking", "electricity services cooking", "Cooking"),
    ("total services", "electricity services", "Services total"),
]

required_cols = [c for pair in pairs for c in pair[:2]]
missing_cols = [c for c in required_cols if c not in df_country.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

plot_df = (
    df_country.groupby("year", dropna=False)[required_cols]
    .sum()
    .sort_index()
)

x = np.arange(len(plot_df.index))
width = 0.38

# 70% overlap between Total and Electricity bars
overlap = 0.70
center_shift = width * (1 - overlap) / 2

# Shared y-scale across all panels
ymax = max(plot_df[required_cols].max()) * 1.05

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (total_col, elec_col, title) in zip(axes, pairs):
    ax.bar(x - center_shift, plot_df[total_col].values, width=width, color="#4E79A7", alpha=0.8, label="Total")
    ax.bar(x + center_shift, plot_df[elec_col].values, width=width, color="#F28E2B", alpha=0.8, label="Electricity")

    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("Value")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index.astype(str), rotation=45)
    ax.set_ylim(0, ymax)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

fig.suptitle(f"{country} services demand components by year", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### `agriculture`

In [ ]:
#################### Parameters
country = "ES"
year_start = 1990
year_end = 2024

#################### 4-panel bar chart: electricity, heat, machinery, and total agriculture (total only)

if "country" not in df_energy_totals.columns or "year" not in df_energy_totals.columns:
    raise KeyError("The DataFrame must contain 'country' and 'year' columns.")

df_country = df_energy_totals[df_energy_totals["country"] == country].copy()
if df_country.empty:
    raise ValueError(f"No rows found for country == '{country}'.")

df_country = df_country[(df_country["year"] >= year_start) & (df_country["year"] <= year_end)].copy()
if df_country.empty:
    raise ValueError(f"No rows found for {country} in the selected period [{year_start}, {year_end}].")

cols = [
    ("total agriculture electricity", "Electricity"),
    ("total agriculture heat", "Heat"),
    ("total agriculture machinery", "Machinery"),
    ("total agriculture", "Agriculture total"),
]

required_cols = [c for c, _ in cols]
missing_cols = [c for c in required_cols if c not in df_country.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

plot_df = (
    df_country.groupby("year", dropna=False)[required_cols]
    .sum()
    .sort_index()
)

x = np.arange(len(plot_df.index))
width = 0.62

# Shared y-scale across all panels
ymax = max(plot_df[required_cols].max()) * 1.05

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (col, title) in zip(axes, cols):
    ax.bar(x, plot_df[col].values, width=width, color="#4E79A7", alpha=0.85, label="Total")

    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("Value")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index.astype(str), rotation=45)
    ax.set_ylim(0, ymax)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

fig.suptitle(f"{country} agriculture demand components by year", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### `aviation`

In [ ]:
#################### Parameters
country = "ES"
year_start = 1990
year_end = 2024

#################### Stacked bar chart: aviation demand components

if "country" not in df_energy_totals.columns or "year" not in df_energy_totals.columns:
    raise KeyError("The DataFrame must contain 'country' and 'year' columns.")

df_country = df_energy_totals[df_energy_totals["country"] == country].copy()
if df_country.empty:
    raise ValueError(f"No rows found for country == '{country}'.")

df_country = df_country[(df_country["year"] >= year_start) & (df_country["year"] <= year_end)].copy()
if df_country.empty:
    raise ValueError(f"No rows found for {country} in the selected period [{year_start}, {year_end}].")

aviation_cols = [
    "total domestic aviation passenger",
    "total international aviation passenger",
    "total domestic aviation freight",
    "total international aviation freight",
]

missing_cols = [c for c in aviation_cols if c not in df_country.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

plot_df = (
    df_country.groupby("year", dropna=False)[aviation_cols]
    .sum()
    .sort_index()
)

ax = plot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(13, 6),
    color=["#4E79A7", "#F28E2B", "#59A14F", "#E15759"],
)

ax.set_title(f"{country} aviation demand components by year")
ax.set_xlabel("Year")
ax.set_ylabel("Value")
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.25)
ax.legend(title="Components", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

### `navigation`

In [ ]:
#################### Parameters
country = "ES"
year_start = 1990
year_end = 2024

#################### Stacked bar chart: navigation demand components

if "country" not in df_energy_totals.columns or "year" not in df_energy_totals.columns:
    raise KeyError("The DataFrame must contain 'country' and 'year' columns.")

df_country = df_energy_totals[df_energy_totals["country"] == country].copy()
if df_country.empty:
    raise ValueError(f"No rows found for country == '{country}'.")

df_country = df_country[(df_country["year"] >= year_start) & (df_country["year"] <= year_end)].copy()
if df_country.empty:
    raise ValueError(f"No rows found for {country} in the selected period [{year_start}, {year_end}].")

navigation_cols = [
    "total domestic navigation",
    "total international navigation",
]

missing_cols = [c for c in navigation_cols if c not in df_country.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

plot_df = (
    df_country.groupby("year", dropna=False)[navigation_cols]
    .sum()
    .sort_index()
)

ax = plot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6),
    color=["#4E79A7", "#F28E2B"],
)

ax.set_title(f"{country} navigation demand components by year")
ax.set_xlabel("Year")
ax.set_ylabel("Value")
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.25)
ax.legend(title="Components", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

### `rail`

In [ ]:
#################### Parameters
country = "ES"
year_start = 1990
year_end = 2024



#################### 3-panel bar chart: rail passenger, rail freight, and rail total

if "country" not in df_energy_totals.columns or "year" not in df_energy_totals.columns:
    raise KeyError("The DataFrame must contain 'country' and 'year' columns.")

df_country = df_energy_totals[df_energy_totals["country"] == country].copy()
if df_country.empty:
    raise ValueError(f"No rows found for country == '{country}'.")

df_country = df_country[(df_country["year"] >= year_start) & (df_country["year"] <= year_end)].copy()
if df_country.empty:
    raise ValueError(f"No rows found for {country} in the selected period [{year_start}, {year_end}].")

pairs = [
    ("total rail passenger", "electricity rail passenger", "Rail passenger"),
    ("total rail freight", "electricity rail freight", "Rail freight"),
    ("total rail", "electricity rail", "Rail total"),
]

required_cols = [c for pair in pairs for c in pair[:2]]
missing_cols = [c for c in required_cols if c not in df_country.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

plot_df = (
    df_country.groupby("year", dropna=False)[required_cols]
    .sum()
    .sort_index()
)

x = np.arange(len(plot_df.index))
width = 0.38

# 70% overlap between Total and Electricity bars
overlap = 0.70
center_shift = width * (1 - overlap) / 2

# Shared y-scale across all panels
ymax = max(plot_df[required_cols].max()) * 1.05

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (total_col, elec_col, title) in zip(axes, pairs):
    ax.bar(x - center_shift, plot_df[total_col].values, width=width, color="#4E79A7", alpha=0.8, label="Total")
    ax.bar(x + center_shift, plot_df[elec_col].values, width=width, color="#F28E2B", alpha=0.8, label="Electricity")

    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("Value")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index.astype(str), rotation=45)
    ax.set_ylim(0, ymax)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

# Hide the unused 4th subplot (only 3 rail components requested)
for ax in axes[len(pairs):]:
    ax.axis("off")

fig.suptitle(f"{country} rail demand components by year", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()